# Object-Oriented Python

Python's object model goes well beyond classes and inheritance. Properties let you add logic to attribute access without changing the calling code. Decorators let you compose and reuse behaviour cleanly. Dataclasses eliminate boilerplate. Abstract base classes enforce contracts across a hierarchy. Together these features make it possible to write Python that is both flexible and self-documenting.

**What's inside:** classes, inheritance, decorators, `@property`, `@classmethod`, `@staticmethod`, dunder methods, dataclasses, enums, and abstract base classes.

**Learn more:** [Python Classes](https://docs.python.org/3/tutorial/classes.html)

## 1. Defining Classes

### 1.1 \_\_init\_\_ and instance methods

In [ ]:
class Dog:
    def __init__(self, name, breed):
        self.name = name
        self.breed = breed

    def speak(self):
        return f'{self.name} says woof!'

rex = Dog('Rex', 'Labrador')
rex.speak()

### 1.2 Class variables vs instance variables

Class variables are shared by all instances; instance variables are unique to each.

In [ ]:
class Dog:
    species = 'Canis lupus familiaris'   # shared by all instances

    def __init__(self, name):
        self.name = name                  # unique per instance

d1 = Dog('Rex')
d2 = Dog('Spot')
print(Dog.species)
print(d1.name, d2.name)

## 2. Inheritance and super()

### 2.1 Basic inheritance

A subclass inherits all methods of its parent and can override them.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f'{self.name} makes a sound.'

class Cat(Animal):
    def speak(self):            # override parent method
        return f'{self.name} says meow!'

Cat('Whiskers').speak()

### 2.2 super(): extend the parent rather than replace it

In [ ]:
class Pet(Animal):
    def __init__(self, name, owner):
        super().__init__(name)    # run Animal.__init__
        self.owner = owner

    def speak(self):
        base = super().speak()    # call parent version too
        return f'{base} (owned by {self.owner})'

Pet('Buddy', 'Alice').speak()

### 2.3 isinstance and issubclass

In [ ]:
p = Pet('Buddy', 'Alice')
print(isinstance(p, Animal))      # True: Pet is a subclass of Animal
print(issubclass(Pet, Animal))

## 3. Decorators

A decorator is a function that takes a function and returns a new (usually enhanced) function.

### 3.1 How decorators work

The `@decorator` syntax is shorthand for `fn = decorator(fn)`.

In [ ]:
# explicit long form (what @ does under the hood)
def shout(fn):
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        return result.upper()
    return wrapper

def greet(name):
    return f'hello, {name}'

greet = shout(greet)   # manual wrapping
greet('world')

In [ ]:
# same thing with @ syntax
@shout
def greet(name):
    return f'hello, {name}'

greet('world')

### 3.2 Preserving metadata with functools.wraps

Without `@wraps`, the wrapper replaces the original function's `__name__` and `__doc__`.

In [ ]:
from functools import wraps

def shout(fn):
    @wraps(fn)               # copies __name__, __doc__, etc. from fn onto wrapper
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs).upper()
    return wrapper

@shout
def greet(name):
    '''Return a greeting.'''
    return f'hello, {name}'

print(greet.__name__)    # 'greet', not 'wrapper'
print(greet.__doc__)

### 3.3 Stacking decorators

Decorators are applied bottom-up; the one closest to the function runs first.

In [ ]:
def bold(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        return '<b>' + fn(*args, **kwargs) + '</b>'
    return wrapper

def italic(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        return '<i>' + fn(*args, **kwargs) + '</i>'
    return wrapper

@bold
@italic          # italic applied first, then bold wraps the result
def greet(name):
    return f'hello, {name}'

greet('world')   # '<b><i>hello, world</i></b>'

### 3.4 Decorator with arguments

Add an outer function that accepts arguments and returns the actual decorator.

In [ ]:
def repeat(n):
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            for _ in range(n - 1):
                fn(*args, **kwargs)
            return fn(*args, **kwargs)
        return wrapper
    return decorator

@repeat(3)
def say(msg):
    print(msg)

say('hello')

## 4. @property

Define computed attributes that look like regular attributes but run code when accessed.

### 4.1 Getter

In [ ]:
import math

class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def area(self):
        return math.pi * self._radius ** 2

c = Circle(5)
c.area   # accessed like an attribute, not a method call

### 4.2 Setter with validation

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius       # routes through the setter

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError('radius must be non-negative')
        self._radius = value

c = Circle(3)
c.radius = 10
print(c.radius)

## 5. @classmethod and @staticmethod

### 5.1 @classmethod: alternative constructors

`cls` receives the class itself, making it easy to define multiple ways to construct an object.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    @classmethod
    def from_tuple(cls, t):
        return cls(*t)

    @classmethod
    def origin(cls):
        return cls(0, 0)

    def __repr__(self):
        return f'Point({self.x}, {self.y})'

print(Point.from_tuple((3, 4)))
print(Point.origin())

### 5.2 @staticmethod: namespaced utility functions

No `self` or `cls`; just a regular function that lives on the class for organisational reasons.

In [ ]:
class MathHelper:
    @staticmethod
    def clamp(value, lo, hi):
        return max(lo, min(hi, value))

MathHelper.clamp(15, 0, 10)

## 6. Dunder (Magic) Methods

Special methods that Python calls automatically when using operators or built-in functions.

### 6.1 \_\_repr\_\_ and \_\_str\_\_

`__repr__` is for developers (unambiguous); `__str__` is for end users (readable).

In [ ]:
class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages

    def __repr__(self):
        return f'Book({self.title!r}, {self.pages})'

    def __str__(self):
        return f'{self.title!r} ({self.pages} pages)'

b = Book('Dune', 412)
print(repr(b))
print(str(b))

### 6.2 \_\_len\_\_ and \_\_eq\_\_

In [ ]:
class Playlist:
    def __init__(self, tracks):
        self.tracks = tracks

    def __len__(self):
        return len(self.tracks)

    def __eq__(self, other):
        return self.tracks == other.tracks

p1 = Playlist(['A', 'B', 'C'])
p2 = Playlist(['A', 'B', 'C'])
print(len(p1))
print(p1 == p2)

### 6.3 \_\_lt\_\_: enables sorting

In [ ]:
class Card:
    def __init__(self, value):
        self.value = value

    def __repr__(self):
        return f'Card({self.value})'

    def __lt__(self, other):
        return self.value < other.value

hand = [Card(7), Card(2), Card(10)]
sorted(hand)

### 6.4 \_\_contains\_\_ and \_\_getitem\_\_: enables `in` and indexing

In [ ]:
class Deck:
    def __init__(self, cards):
        self.cards = cards

    def __contains__(self, item):
        return item in self.cards

    def __getitem__(self, index):
        return self.cards[index]

d = Deck([1, 2, 3, 4])
print(3 in d)
print(d[0])

## 7. Dataclasses

`@dataclass` generates `__init__`, `__repr__`, and `__eq__` automatically.

Ref: https://docs.python.org/3/library/dataclasses.html

### 7.1 Basic @dataclass

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

p = Point(1.5, 2.5)
print(p)     # __repr__ generated for free
print(p.x)

### 7.2 Default values and field()

Mutable defaults (lists, dicts) must use `field(default_factory=...)` to avoid sharing state between instances.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Config:
    host: str = 'localhost'
    port: int = 8080
    tags: list = field(default_factory=list)   # safe mutable default

Config()

### 7.3 \_\_post_init\_\_: computed fields

In [ ]:
import math
from dataclasses import dataclass, field

@dataclass
class Circle:
    radius: float
    area: float = field(init=False)    # excluded from __init__

    def __post_init__(self):
        self.area = round(math.pi * self.radius ** 2, 2)

Circle(5)

### 7.4 frozen=True: immutable dataclass

Frozen instances are hashable and can be used as dict keys or in sets.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Color:
    r: int
    g: int
    b: int

red = Color(255, 0, 0)
print(red)
# red.r = 100  # raises FrozenInstanceError

## 8. Enums

Enums give names to a fixed set of values, making code more readable and preventing invalid states.

Ref: https://docs.python.org/3/library/enum.html

### 8.1 Basic Enum

In [ ]:
from enum import Enum

class Direction(Enum):
    NORTH = 'N'
    SOUTH = 'S'
    EAST  = 'E'
    WEST  = 'W'

d = Direction.NORTH
print(d)
print(d.name)
print(d.value)

### 8.2 auto(): automatically assigned values

In [ ]:
from enum import Enum, auto

class Status(Enum):
    PENDING  = auto()
    RUNNING  = auto()
    DONE     = auto()
    FAILED   = auto()

print(Status.RUNNING)
print(Status.RUNNING.value)

### 8.3 IntEnum: interoperable with int

In [ ]:
from enum import IntEnum

class Priority(IntEnum):
    LOW    = 1
    MEDIUM = 2
    HIGH   = 3

# IntEnum members compare directly with ints
print(Priority.HIGH > 2)
print(Priority.HIGH == 3)

### 8.4 Using enums in conditionals

Comparing against enum members is safer than comparing against raw strings or integers.

In [ ]:
def handle(status):
    if status == Status.DONE:
        return 'all finished'
    if status == Status.FAILED:
        return 'something went wrong'
    return 'still working...'

handle(Status.DONE)

### 8.5 Iterating members

In [ ]:
for member in Direction:
    print(f'{member.name}: {member.value}')

### 8.6 Lookup by value

In [ ]:
Direction('S')   # get a member from its value

## 9. Abstract Base Classes

ABCs define an interface contract; subclasses *must* implement certain methods or Python raises a `TypeError` at instantiation.

Ref: https://docs.python.org/3/library/abc.html

### 9.1 Defining an abstract class

A class with at least one `@abstractmethod` cannot be instantiated directly.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        ...

    @abstractmethod
    def perimeter(self) -> float:
        ...

# Shape()  # TypeError: Can't instantiate abstract class

### 9.2 Enforcing an interface

A subclass that implements all abstract methods can be instantiated; one that doesn't raises a `TypeError`.

In [ ]:
import math

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius

c = Circle(5)
print(round(c.area(), 2))
print(round(c.perimeter(), 2))

In [ ]:
# a subclass that forgets to implement a method
class Blob(Shape):
    def area(self):
        return 0

try:
    Blob()
except TypeError as e:
    print(e)

### 9.3 Abstract properties

Combine `@property` and `@abstractmethod` to require a computed attribute in every subclass.

In [ ]:
class Animal(ABC):
    @property
    @abstractmethod
    def sound(self) -> str:
        ...

    def speak(self):
        return f'{self.__class__.__name__} says {self.sound}'

class Dog(Animal):
    @property
    def sound(self):
        return 'woof'

Dog().speak()